# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets by their @id
record_sets = dataset.metadata.record_sets
print('Available record sets:')
for r in record_sets:
    print(f"- @id: {r['@id']} -- name: {r.get('name', '')}")

# For each record set, list its fields by @id
for r in record_sets:
    print(f"\nRecord set @id: {r['@id']}, name: {r.get('name','')}")
    if 'fields' in r:
        print('  Fields:')
        for f in r['fields']:
            fid = f['@id'] if isinstance(f, dict) and '@id' in f else f
            print(f"    - @id: {fid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, pick the first record set for extraction.
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]['@id']
    record_sets_ids = [r['@id'] for r in record_sets]
    print(f"Extracting data from record sets: {record_sets_ids}")

    dataframes = {}
    for record_set_id in record_sets_ids:
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame shape for {record_set_id}: {df.shape}")

    print(f"\nColumns in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets are defined in the dataset!')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: operate on a numeric field if present.
import numpy as np
if len(record_sets) > 0:
    df = dataframes[main_record_set_id]
    # Attempt to guess a numeric field
    # Croissant schema usually specifies 'fields' with their @id and name
    sample_numeric = None
    for f in dataset.metadata.record_sets[0]['fields']:
        field_meta = f if isinstance(f, dict) else dataset.metadata.fields_by_id[f]
        # Try typical protein dataset columns
        fname = field_meta.get('name', field_meta.get('@id',''))
        fid = field_meta.get('@id', fname)
        # Search for fields likely to be numeric
        possible_keywords = ['count', 'peptide', 'coverage', 'abundance', 'mw', 'weight']
        if any(k in fname.lower() for k in possible_keywords):
            # Check if column exists and is numeric
            col_candidates = [fid, fname]
            for c in col_candidates:
                if c in df.columns and np.issubdtype(df[c].dtype, np.number):
                    sample_numeric = c
                    sample_numeric_id = fid
                    break
        if sample_numeric:
            break

    if sample_numeric:
        # Filtering, normalization, grouping
        threshold = float(df[sample_numeric].mean()) if df[sample_numeric].dtype != 'O' else 10
        filtered_df = df[df[sample_numeric] > threshold]
        print(f"Filtered records with {sample_numeric} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{sample_numeric}_normalized"] = (filtered_df[sample_numeric] - filtered_df[sample_numeric].mean()) / filtered_df[sample_numeric].std()
        print(f"Normalized {sample_numeric} for filtered records:")
        print(filtered_df[[sample_numeric, f"{sample_numeric}_normalized"]].head())

        # Try grouping by a second field, e.g. description or category
        group_field = None
        for f in dataset.metadata.record_sets[0]['fields']:
            field_meta = f if isinstance(f, dict) else dataset.metadata.fields_by_id[f]
            fname = field_meta.get('name', field_meta.get('@id',''))
            fid = field_meta.get('@id', fname)
            if fname != sample_numeric and fid in df.columns and df[fid].dtype == object:
                group_field = fid
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No obvious numeric field found for EDA.")
else:
    print('No record sets available for analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram and boxplot for the detected numeric field
if len(record_sets) > 0 and 'sample_numeric' in locals() and sample_numeric in df.columns and np.issubdtype(df[sample_numeric].dtype, np.number):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(df[sample_numeric].dropna(), kde=True)
    plt.title(f'Histogram of {sample_numeric}')
    plt.subplot(1,2,2)
    sns.boxplot(x=df[sample_numeric].dropna())
    plt.title(f'Boxplot of {sample_numeric}')
    plt.tight_layout()
    plt.show()

    # If a group field was found and not too many groups, show a violinplot
    if 'group_field' in locals() and group_field and group_field in df.columns and df[group_field].nunique() < 10:
        plt.figure(figsize=(7,4))
        sns.violinplot(x=df[group_field], y=df[sample_numeric])
        plt.title(f'{sample_numeric} distribution by {group_field}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*The above analysis demonstrates basic usage of the `mlcroissant` library for FAIR dataset access: listing record sets and fields using their `@id`, loading records to pandas DataFrames, and conducting exploratory analysis. Further scientific analysis may be performed as needed using the domain-specific fields listed per their IDs in previous sections.*